In [1]:
# ===== REDEFINIR FUNCIÓN create_trading_chart CORREGIDA =====
def create_trading_chart(data, signals=None, title="NAS100 ICT Analysis"):
    """Crea gráfico de trading con señales ICT - versión corregida"""
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=(title, 'Volume'),
        row_heights=[0.8, 0.2]
    )
    
    # Candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data['Open'],
            high=data['High'],
            low=data['Low'],
            close=data['Close'],
            name='NAS100'
        ),
        row=1, col=1
    )
    
    # EMAs si están disponibles
    if 'EMA_9' in data.columns:
        fig.add_trace(
            go.Scatter(x=data.index, y=data['EMA_9'], name='EMA 9', line=dict(color='orange', width=1)),
            row=1, col=1
        )
    if 'EMA_21' in data.columns:
        fig.add_trace(
            go.Scatter(x=data.index, y=data['EMA_21'], name='EMA 21', line=dict(color='blue', width=1)),
            row=1, col=1
        )
    if 'EMA_50' in data.columns:
        fig.add_trace(
            go.Scatter(x=data.index, y=data['EMA_50'], name='EMA 50', line=dict(color='red', width=1)),
            row=1, col=1
        )
    
    # Killzones (solo si el índice es datetime)
    try:
        if hasattr(data.index, 'date'):
            # Definir horarios de killzones
            asia_start = datetime.strptime('00:00', '%H:%M').time()
            asia_end = datetime.strptime('03:00', '%H:%M').time()
            london_start = datetime.strptime('07:00', '%H:%M').time()
            london_end = datetime.strptime('10:00', '%H:%M').time()
            ny_start = datetime.strptime('13:00', '%H:%M').time()
            ny_end = datetime.strptime('16:00', '%H:%M').time()
            
            # Añadir killzones como shapes
            for date in pd.date_range(data.index.date.min(), data.index.date.max()):
                # Asia Killzone
                fig.add_vrect(
                    x0=pd.Timestamp.combine(date, asia_start),
                    x1=pd.Timestamp.combine(date, asia_end),
                    fillcolor="yellow", opacity=0.1,
                    layer="below", line_width=0,
                    row=1, col=1
                )
                
                # London Killzone
                fig.add_vrect(
                    x0=pd.Timestamp.combine(date, london_start),
                    x1=pd.Timestamp.combine(date, london_end),
                    fillcolor="blue", opacity=0.1,
                    layer="below", line_width=0,
                    row=1, col=1
                )
                
                # NY Killzone
                fig.add_vrect(
                    x0=pd.Timestamp.combine(date, ny_start),
                    x1=pd.Timestamp.combine(date, ny_end),
                    fillcolor="green", opacity=0.1,
                    layer="below", line_width=0,
                    row=1, col=1
                )
    except (AttributeError, TypeError):
        print("ℹ️ Killzones omitidas: índice no es de tipo datetime")
    
    # Añadir señales si están disponibles
    if signals and len(signals) > 0:
        for signal in signals:
            color = 'green' if signal.get('type', signal.get('direction')) == 'LONG' else 'red'
            symbol = 'triangle-up' if signal.get('type', signal.get('direction')) == 'LONG' else 'triangle-down'
            
            fig.add_trace(
                go.Scatter(
                    x=[signal['time']],
                    y=[signal['price']],
                    mode='markers',
                    marker=dict(
                        symbol=symbol,
                        size=15,
                        color=color,
                        line=dict(width=2, color='white')
                    ),
                    name=f"{signal.get('type', signal.get('direction', 'Signal'))} Signal",
                    hovertemplate=f"<b>{signal.get('type', signal.get('direction', 'Signal'))}</b><br>" +
                                f"Tiempo: %{{x}}<br>" +
                                f"Precio: {signal['price']:.2f}<br>" +
                                f"Fuerza: {signal.get('strength', 0):.1f}%<extra></extra>"
                ),
                row=1, col=1
            )
    
    # Volume chart - usar tick_volume si Volume no existe
    volume_column = 'Volume' if 'Volume' in data.columns else 'tick_volume'
    
    if volume_column in data.columns:
        colors = ['red' if close < open else 'green' for close, open in zip(data['Close'], data['Open'])]
        fig.add_trace(
            go.Bar(x=data.index, y=data[volume_column], name='Volume', marker_color=colors),
            row=2, col=1
        )
    
    # Layout
    fig.update_layout(
        title=title,
        yaxis_title='Precio',
        xaxis_title='Tiempo',
        template='plotly_dark',
        showlegend=True,
        height=800,
        hovermode='x unified'
    )
    
    return fig

print("✅ Función create_trading_chart redefinida con correcciones")

✅ Función create_trading_chart redefinida con correcciones


In [2]:
# ===== PROCESADOR DE FECHAS DINÁMICO =====
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re

def process_date_input(date_input):
    """
    Procesa diferentes formatos de fecha y devuelve configuración completa
    
    Formatos aceptados:
    - '16092025' -> 16 de septiembre 2025
    - '16/09/2025' -> 16 de septiembre 2025
    - '2025-09-16' -> 16 de septiembre 2025
    """
    try:
        # Eliminar espacios y caracteres especiales
        date_str = str(date_input).strip()
        
        # Formato DDMMYYYY (ej: 16092025)
        if re.match(r'^\d{8}$', date_str):
            day = int(date_str[:2])
            month = int(date_str[2:4])
            year = int(date_str[4:8])
            target_date = datetime(year, month, day)
            
        # Formato DD/MM/YYYY
        elif re.match(r'^\d{1,2}/\d{1,2}/\d{4}$', date_str):
            parts = date_str.split('/')
            day, month, year = int(parts[0]), int(parts[1]), int(parts[2])
            target_date = datetime(year, month, day)
            
        # Formato YYYY-MM-DD
        elif re.match(r'^\d{4}-\d{1,2}-\d{1,2}$', date_str):
            target_date = datetime.strptime(date_str, '%Y-%m-%d')
            
        else:
            raise ValueError(f"Formato de fecha no reconocido: {date_input}")
        
        # Crear configuración completa
        config = {
            'symbol': 'EURUSD',
            'timeframe': '15m',
            'timezone': 'UTC',
            'target_date': target_date.strftime('%Y-%m-%d'),
            'analysis_start': '04:00',
            'analysis_end': '20:00',
            'formatted_date': target_date.strftime('%d/%m/%Y'),
            'day_name': target_date.strftime('%A'),
            'month_name': target_date.strftime('%B')
        }
        
        return target_date, config
        
    except Exception as e:
        raise ValueError(f"Error procesando fecha {date_input}: {str(e)}")

# ===== CONFIGURACIÓN DINÁMICA =====
# 🔧 CAMBIA ESTA FECHA PARA ANALIZAR CUALQUIER DÍA
INPUT_DATE = "29092025"  # Formato: DDMMYYYY - CAMBIAR AQUÍ PARA NUEVA FECHA

try:
    target_datetime, DYNAMIC_CONFIG = process_date_input(INPUT_DATE)
    
    print("🎯 CONFIGURACIÓN DINÁMICA GENERADA")
    print("=" * 50)
    print(f"📅 Fecha objetivo: {DYNAMIC_CONFIG['formatted_date']} ({DYNAMIC_CONFIG['day_name']})")
    print(f"📊 Símbolo: {DYNAMIC_CONFIG['symbol']}")
    print(f"⏰ Timeframe: {DYNAMIC_CONFIG['timeframe']}")
    print(f"🌍 Timezone: {DYNAMIC_CONFIG['timezone']}")
    print(f"⏰ Horario análisis: {DYNAMIC_CONFIG['analysis_start']} - {DYNAMIC_CONFIG['analysis_end']} UTC")
    print(f"📆 Mes: {DYNAMIC_CONFIG['month_name']} {target_datetime.year}")
    
    # Verificar si es día hábil
    if target_datetime.weekday() >= 5:
        print("⚠️  ADVERTENCIA: La fecha seleccionada cae en fin de semana")
        print("   Los mercados Forex están cerrados los fines de semana")
    else:
        print("✅ Fecha válida para análisis (día hábil)")
        
except Exception as e:
    print(f"❌ ERROR: {str(e)}")
    print("💡 Formatos válidos: '16092025', '16/09/2025', '2025-09-16'")
    
    # Configuración por defecto en caso de error
    DYNAMIC_CONFIG = {
        'symbol': 'NAS100',
        'timeframe': '15m',
        'timezone': 'UTC',
        'target_date': '2025-09-19',
        'analysis_start': '04:00',
        'analysis_end': '20:00',
        'formatted_date': '19/09/2025',
        'day_name': 'Friday',
        'month_name': 'September'
    }

🎯 CONFIGURACIÓN DINÁMICA GENERADA
📅 Fecha objetivo: 29/09/2025 (Monday)
📊 Símbolo: EURUSD
⏰ Timeframe: 15m
🌍 Timezone: UTC
⏰ Horario análisis: 04:00 - 20:00 UTC
📆 Mes: September 2025
✅ Fecha válida para análisis (día hábil)


# 📊 ICT Trading Analysis - 19 Septiembre 2025
Análisis completo del NAS100 usando metodología Inner Circle Trader (ICT)

In [3]:
# ===== IMPORTACIONES Y CONFIGURACIÓN DINÁMICA =====
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import pytz
import warnings
warnings.filterwarnings('ignore')

# Usar la configuración dinámica generada en la celda anterior
CONFIG = DYNAMIC_CONFIG.copy()

print("✅ Configuración dinámica aplicada")
print("=" * 40)
print(f"📅 Analizando: {CONFIG['formatted_date']} ({CONFIG['day_name']})")
print(f"📊 Símbolo: {CONFIG['symbol']}")
print(f"⏰ Timeframe: {CONFIG['timeframe']}")
print(f"🌍 Timezone: {CONFIG['timezone']}")
print(f"⏰ Ventana: {CONFIG['analysis_start']} - {CONFIG['analysis_end']} UTC")

# Verificación adicionale
print(f"\n🔍 DETALLES DEL ANÁLISIS:")
print(f"   • Mes: {CONFIG['month_name']}")
print(f"   • Año: {target_datetime.year}")
print(f"   • Día de la semana: {CONFIG['day_name']}")
print(f"   • Fecha ISO: {CONFIG['target_date']}")

✅ Configuración dinámica aplicada
📅 Analizando: 29/09/2025 (Monday)
📊 Símbolo: EURUSD
⏰ Timeframe: 15m
🌍 Timezone: UTC
⏰ Ventana: 04:00 - 20:00 UTC

🔍 DETALLES DEL ANÁLISIS:
   • Mes: September
   • Año: 2025
   • Día de la semana: Monday
   • Fecha ISO: 2025-09-29


In [4]:
# ===== CONEXIÓN MT5 Y DESCARGA DE DATOS DINÁMICA =====
import MetaTrader5 as mt5

print(f"🔌 Conectando a MT5 para descargar datos del {CONFIG['formatted_date']}...")

# Inicializar MT5
if not mt5.initialize():
    print("❌ Error inicializando MT5")
    mt5.shutdown()
else:
    # Mostrar información de la cuenta
    account_info = mt5.account_info()
    if account_info is not None:
        print(f"✅ MT5 Conectado exitosamente")
        print(f"   📊 Servidor: {account_info.server}")
        print(f"   👤 Cuenta: {account_info.login}")
        print(f"   💰 Balance: ${account_info.balance:,.2f}")
    
    # Configurar tiempos para la fecha específica
    target_date_obj = datetime.strptime(CONFIG['target_date'], '%Y-%m-%d')
    start_time = target_date_obj.replace(hour=4, minute=0, second=0, microsecond=0)  # 4 AM UTC
    end_time = target_date_obj.replace(hour=20, minute=0, second=0, microsecond=0)   # 8 PM UTC
    
    print(f"\n📥 Descargando datos de {CONFIG['symbol']}...")
    print(f"   ⏰ Desde: {start_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"   ⏰ Hasta: {end_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"   📊 Timeframe: {CONFIG['timeframe']}")
    
    # Descargar datos
    timeframe = mt5.TIMEFRAME_M15  # 15 minutos
    rates = mt5.copy_rates_range(CONFIG['symbol'], timeframe, start_time, end_time)
    
    if rates is not None and len(rates) > 0:
        print(f"✅ {len(rates)} velas descargadas exitosamente")
        
        # Convertir a DataFrame
        data_dynamic = pd.DataFrame(rates)
        data_dynamic['time'] = pd.to_datetime(data_dynamic['time'], unit='s')
        
        # Ajustar zona horaria
        data_dynamic['time'] = data_dynamic['time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York')
        
        print(f"📊 Rango de datos:")
        print(f"   🕐 Inicio: {data_dynamic['time'].min()}")
        print(f"   🕐 Final: {data_dynamic['time'].max()}")
        print(f"   📈 Precio máximo: {data_dynamic['high'].max():,.2f}")
        print(f"   📉 Precio mínimo: {data_dynamic['low'].min():,.2f}")
        
        # Guardar para análisis posterior
        DYNAMIC_DATA = data_dynamic.copy()
        
    else:
        print("❌ No se pudieron descargar datos")
        print("   Posibles causas:")
        print("   • Fecha fuera del rango disponible")
        print("   • Día no hábil (fin de semana/feriado)")
        print("   • Problemas de conexión")
        DYNAMIC_DATA = None

🔌 Conectando a MT5 para descargar datos del 29/09/2025...
✅ MT5 Conectado exitosamente
   📊 Servidor: VantageInternational-Demo
   👤 Cuenta: 11115654
   💰 Balance: $1,000.00

📥 Descargando datos de EURUSD...
   ⏰ Desde: 2025-09-29 04:00:00 UTC
   ⏰ Hasta: 2025-09-29 20:00:00 UTC
   📊 Timeframe: 15m
✅ 35 velas descargadas exitosamente
📊 Rango de datos:
   🕐 Inicio: 2025-09-29 05:00:00-04:00
   🕐 Final: 2025-09-29 13:30:00-04:00
   📈 Precio máximo: 1.18
   📉 Precio mínimo: 1.17


In [5]:
# ===== CORRECCIÓN DE FORMATO DE DATOS =====

def standardize_data_format(data):
    """Estandariza el formato de datos para compatibilidad con funciones ICT"""
    
    # Mapeo de columnas (minúsculas a mayúsculas)
    column_mapping = {
        'time': 'time',
        'open': 'Open', 
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'tick_volume': 'tick_volume',
        'spread': 'spread',
        'real_volume': 'real_volume'
    }
    
    # Renombrar columnas
    data_corrected = data.copy()
    for old_name, new_name in column_mapping.items():
        if old_name in data_corrected.columns:
            data_corrected = data_corrected.rename(columns={old_name: new_name})
    
    # Verificar que todas las columnas necesarias estén presentes
    required_columns = ['time', 'Open', 'High', 'Low', 'Close', 'tick_volume']
    missing_columns = [col for col in required_columns if col not in data_corrected.columns]
    
    if missing_columns:
        print(f"⚠️ Columnas faltantes: {missing_columns}")
    else:
        print("✅ Formato de datos estandarizado correctamente")
    
    return data_corrected

# Aplicar corrección si tenemos datos
if 'DYNAMIC_DATA' in locals() and DYNAMIC_DATA is not None:
    print("🔧 Corrigiendo formato de datos para compatibilidad ICT...")
    DYNAMIC_DATA = standardize_data_format(DYNAMIC_DATA)
    print(f"📊 Columnas disponibles: {list(DYNAMIC_DATA.columns)}")
else:
    print("⚠️ No hay datos dinámicos para corregir")

🔧 Corrigiendo formato de datos para compatibilidad ICT...
✅ Formato de datos estandarizado correctamente
📊 Columnas disponibles: ['time', 'Open', 'High', 'Low', 'Close', 'tick_volume', 'spread', 'real_volume']


In [6]:
# ===== ANÁLISIS ICT COMPLETO CON FVG, BPR Y MSS =====

def detect_fvg(data):
    """Detecta Fair Value Gaps (FVG)"""
    fvgs = []
    
    for i in range(1, len(data) - 1):
        prev_candle = data.iloc[i-1]
        current_candle = data.iloc[i]
        next_candle = data.iloc[i+1]
        
        # FVG Bullish: gap entre low actual y high anterior
        if (current_candle['Low'] > prev_candle['High'] and 
            next_candle['High'] > current_candle['Low']):
            
            fvg = {
                'time': current_candle['time'],
                'type': 'BULLISH',
                'high': current_candle['Low'],
                'low': prev_candle['High'],
                'size': current_candle['Low'] - prev_candle['High'],
                'filled': False
            }
            fvgs.append(fvg)
        
        # FVG Bearish: gap entre high actual y low anterior  
        elif (current_candle['High'] < prev_candle['Low'] and
              next_candle['Low'] < current_candle['High']):
            
            fvg = {
                'time': current_candle['time'],
                'type': 'BEARISH', 
                'high': prev_candle['Low'],
                'low': current_candle['High'],
                'size': prev_candle['Low'] - current_candle['High'],
                'filled': False
            }
            fvgs.append(fvg)
    
    return fvgs

def detect_bpr(data):
    """Detecta Break of Previous Range (BPR)"""
    bprs = []
    lookback = 10
    
    for i in range(lookback, len(data)):
        current = data.iloc[i]
        prev_range = data.iloc[i-lookback:i]
        
        range_high = prev_range['High'].max()
        range_low = prev_range['Low'].min()
        
        # BPR Bullish: rompe máximo del rango anterior
        if current['High'] > range_high:
            bpr = {
                'time': current['time'],
                'type': 'BULLISH',
                'level': range_high,
                'strength': (current['High'] - range_high) / range_high * 100
            }
            bprs.append(bpr)
            
        # BPR Bearish: rompe mínimo del rango anterior
        elif current['Low'] < range_low:
            bpr = {
                'time': current['time'],
                'type': 'BEARISH', 
                'level': range_low,
                'strength': (range_low - current['Low']) / range_low * 100
            }
            bprs.append(bpr)
    
    return bprs

def detect_mss(data):
    """Detecta Market Structure Shifts (MSS)"""
    mss_events = []
    
    # Buscar swing highs y swing lows
    swing_points = []
    lookback = 5
    
    for i in range(lookback, len(data) - lookback):
        current = data.iloc[i]
        
        # Swing High
        is_swing_high = True
        for j in range(1, lookback + 1):
            if (data.iloc[i-j]['High'] >= current['High'] or 
                data.iloc[i+j]['High'] >= current['High']):
                is_swing_high = False
                break
        
        if is_swing_high:
            swing_points.append({
                'time': current['time'],
                'type': 'HIGH',
                'price': current['High'],
                'index': i
            })
        
        # Swing Low
        is_swing_low = True  
        for j in range(1, lookback + 1):
            if (data.iloc[i-j]['Low'] <= current['Low'] or
                data.iloc[i+j]['Low'] <= current['Low']):
                is_swing_low = False
                break
                
        if is_swing_low:
            swing_points.append({
                'time': current['time'],
                'type': 'LOW',
                'price': current['Low'],
                'index': i
            })
    
    # Detectar cambios de estructura
    swing_points.sort(key=lambda x: x['index'])
    
    for i in range(1, len(swing_points)):
        current_swing = swing_points[i]
        prev_swing = swing_points[i-1]
        
        # MSS Bullish: Low más alto después de High más alto
        if (prev_swing['type'] == 'HIGH' and current_swing['type'] == 'LOW' and
            i > 1 and swing_points[i-2]['type'] == 'LOW' and
            current_swing['price'] > swing_points[i-2]['price']):
            
            mss_events.append({
                'time': current_swing['time'],
                'type': 'BULLISH',
                'description': 'Higher Low after Higher High',
                'strength': 75
            })
            
        # MSS Bearish: High más bajo después de Low más bajo  
        elif (prev_swing['type'] == 'LOW' and current_swing['type'] == 'HIGH' and
              i > 1 and swing_points[i-2]['type'] == 'HIGH' and
              current_swing['price'] < swing_points[i-2]['price']):
            
            mss_events.append({
                'time': current_swing['time'],
                'type': 'BEARISH',
                'description': 'Lower High after Lower Low',
                'strength': 75
            })
    
    return mss_events

def simple_ict_analysis():
    """Análisis ICT completo para la fecha dinámica"""
    
    if DYNAMIC_DATA is None:
        print("❌ No hay datos disponibles para analizar")
        return None
    
    print(f"🔍 Análisis ICT completo para {CONFIG['formatted_date']}...")
    print("=" * 60)
    
    try:
        data = DYNAMIC_DATA.copy()
        
        # 1. DETECTAR ELEMENTOS ICT
        print("🔄 Detectando Fair Value Gaps (FVG)...")
        fvgs = detect_fvg(data)
        print(f"   ✅ {len(fvgs)} FVGs detectados")
        
        print("🔄 Detectando Break of Previous Range (BPR)...")
        bprs = detect_bpr(data)
        print(f"   ✅ {len(bprs)} BPRs detectados")
        
        print("🔄 Detectando Market Structure Shifts (MSS)...")
        mss_events = detect_mss(data)
        print(f"   ✅ {len(mss_events)} MSS detectados")
        
        # 2. DETECTAR SWEEPS Y GENERAR SEÑALES
        print("🔄 Detectando Sweeps y generando señales...")
        sweeps_simple = []
        signals_found = []
        
        for i in range(20, len(data)-5):
            current = data.iloc[i]
            prev_highs = data.iloc[i-20:i]['High'].values
            prev_lows = data.iloc[i-20:i]['Low'].values
            
            # Detectar sweep de máximos (potencial LONG)
            if current['High'] > np.max(prev_highs):
                # Verificar si es un fakeout (precio regresa)
                next_candles = data.iloc[i+1:i+6]
                if len(next_candles) >= 3:
                    avg_close = next_candles['Close'].mean()
                    if avg_close < current['High'] * 0.999:  # Fakeout confirmado
                        
                        # Calcular fuerza de la señal
                        price_move = (current['High'] - np.max(prev_highs)) / np.max(prev_highs) * 100
                        volume_factor = current['tick_volume'] / data.iloc[i-20:i]['tick_volume'].mean()
                        
                        strength = min(95, 65 + price_move * 1000 + volume_factor * 5)
                        
                        if strength >= 65:
                            signal = {
                                'time': current['time'],
                                'direction': 'LONG',
                                'price': current['High'],
                                'strength': round(strength, 1),
                                'type': 'SWEEP_HIGH'
                            }
                            signals_found.append(signal)
            
            # Detectar sweep de mínimos (potencial SHORT)
            elif current['Low'] < np.min(prev_lows):
                # Verificar si es un fakeout (precio regresa)
                next_candles = data.iloc[i+1:i+6]
                if len(next_candles) >= 3:
                    avg_close = next_candles['Close'].mean()
                    if avg_close > current['Low'] * 1.001:  # Fakeout confirmado
                        
                        # Calcular fuerza de la señal
                        price_move = (np.min(prev_lows) - current['Low']) / np.min(prev_lows) * 100
                        volume_factor = current['tick_volume'] / data.iloc[i-20:i]['tick_volume'].mean()
                        
                        strength = min(95, 65 + price_move * 1000 + volume_factor * 5)
                        
                        if strength >= 65:
                            signal = {
                                'time': current['time'],
                                'direction': 'SHORT',
                                'price': current['Low'],
                                'strength': round(strength, 1),
                                'type': 'SWEEP_LOW'
                            }
                            signals_found.append(signal)
        
        # Clasificar señales por calidad
        for signal in signals_found:
            if signal['strength'] >= 85:
                signal['quality_tier'] = 'PREMIUM'
                signal['quality_icon'] = '💎'
            elif signal['strength'] >= 75:
                signal['quality_tier'] = 'HIGH'
                signal['quality_icon'] = '🔥'
            elif signal['strength'] >= 65:
                signal['quality_tier'] = 'MEDIUM'
                signal['quality_icon'] = '⚡'
            else:
                signal['quality_tier'] = 'LOW'
                signal['quality_icon'] = '⚠️'
        
        # Contar señales por calidad
        premium_count = len([s for s in signals_found if s['quality_tier'] == 'PREMIUM'])
        high_count = len([s for s in signals_found if s['quality_tier'] == 'HIGH'])
        medium_count = len([s for s in signals_found if s['quality_tier'] == 'MEDIUM'])
        low_count = len([s for s in signals_found if s['quality_tier'] == 'LOW'])
        
        # Clasificar el día
        if premium_count >= 2:
            day_classification = "🔥 EXCELENTE"
        elif premium_count >= 1 or high_count >= 2:
            day_classification = "✅ BUENO"
        elif high_count >= 1 or medium_count >= 2:
            day_classification = "⚠️ REGULAR"
        else:
            day_classification = "❌ MALO"
        
        # Mostrar resultados
        print(f"📊 RESULTADOS DEL ANÁLISIS:")
        print(f"   📅 Fecha: {CONFIG['formatted_date']} ({CONFIG['day_name']})")
        print(f"   🏆 Clasificación: {day_classification}")
        print(f"   🎯 Total señales: {len(signals_found)}")
        print(f"   📊 Calidad: P:{premium_count}, H:{high_count}, M:{medium_count}, L:{low_count}")
        
        if signals_found:
            avg_strength = sum(s['strength'] for s in signals_found) / len(signals_found)
            print(f"   💪 Fuerza promedio: {avg_strength:.1f}%")
            
            print(f"\n📋 DETALLE DE SEÑALES:")
            for i, signal in enumerate(signals_found, 1):
                direction_icon = "🟢" if signal['direction'] == 'LONG' else "🔴"
                time_str = signal['time'].strftime('%H:%M')
                print(f"   {i}. {direction_icon} {signal['direction']} {signal['quality_icon']} {signal['quality_tier']}")
                print(f"      ⏰ {time_str} | 💰 {signal['price']:,.2f} | 💪 {signal['strength']}%")
        else:
            print(f"   ❌ No se encontraron señales de calidad suficiente")
        
        # Mostrar resumen de elementos ICT detectados
        print(f"\n📊 ELEMENTOS ICT DETECTADOS:")
        print(f"   🔶 Fair Value Gaps: {len(fvgs)} ({'Bullish: ' + str(len([f for f in fvgs if f['type'] == 'BULLISH'])) + ', Bearish: ' + str(len([f for f in fvgs if f['type'] == 'BEARISH']))})")
        print(f"   📈 Break Previous Range: {len(bprs)} ({'Bullish: ' + str(len([b for b in bprs if b['type'] == 'BULLISH'])) + ', Bearish: ' + str(len([b for b in bprs if b['type'] == 'BEARISH']))})")
        print(f"   🔄 Market Structure Shifts: {len(mss_events)} ({'Bullish: ' + str(len([m for m in mss_events if m['type'] == 'BULLISH'])) + ', Bearish: ' + str(len([m for m in mss_events if m['type'] == 'BEARISH']))})")

        return {
            'data': data,
            'signals': signals_found,
            'fvgs': fvgs,
            'bprs': bprs, 
            'mss': mss_events,
            'classification': day_classification,
            'stats': {
                'premium': premium_count,
                'high': high_count,
                'medium': medium_count,
                'low': low_count
            }
        }
        
    except Exception as e:
        print(f"❌ Error en el análisis: {str(e)}")
        return None

# Ejecutar análisis simplificado
SIMPLE_RESULTS = simple_ict_analysis()

🔍 Análisis ICT completo para 29/09/2025...
🔄 Detectando Fair Value Gaps (FVG)...
   ✅ 0 FVGs detectados
🔄 Detectando Break of Previous Range (BPR)...
   ✅ 10 BPRs detectados
🔄 Detectando Market Structure Shifts (MSS)...
   ✅ 0 MSS detectados
🔄 Detectando Sweeps y generando señales...
📊 RESULTADOS DEL ANÁLISIS:
   📅 Fecha: 29/09/2025 (Monday)
   🏆 Clasificación: ❌ MALO
   🎯 Total señales: 0
   📊 Calidad: P:0, H:0, M:0, L:0
   ❌ No se encontraron señales de calidad suficiente

📊 ELEMENTOS ICT DETECTADOS:
   🔶 Fair Value Gaps: 0 (Bullish: 0, Bearish: 0)
   📈 Break Previous Range: 10 (Bullish: 7, Bearish: 3)
   🔄 Market Structure Shifts: 0 (Bullish: 0, Bearish: 0)


In [7]:
# ===== GRÁFICO SIMPLIFICADO DINÁMICO - VERSIÓN CORREGIDA =====

def create_simple_chart():
    """Crea gráfico simplificado para la fecha dinámica"""
    
    if SIMPLE_RESULTS is None:
        print("❌ No hay resultados para graficar")
        return None
    
    print(f"🎨 Creando gráfico para {CONFIG['formatted_date']}...")
    
    data = SIMPLE_RESULTS['data']
    signals = SIMPLE_RESULTS['signals']
    
    # Verificar que tenemos datos válidos
    if data is None or len(data) == 0:
        print("❌ No hay datos válidos para graficar")
        return None
    
    print(f"📊 Datos disponibles: {len(data)} velas")
    print(f"🎯 Señales disponibles: {len(signals)}")
    
    # Crear gráfico básico
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=[
            f'{CONFIG["symbol"]} - {CONFIG["formatted_date"]} - {SIMPLE_RESULTS["classification"]}',
            'Volumen'
        ],
        row_heights=[0.75, 0.25]
    )
    
    # 1. GRÁFICO DE VELAS - PRIORITARIO
    try:
        fig.add_trace(
            go.Candlestick(
                x=data['time'],
                open=data['Open'],
                high=data['High'],
                low=data['Low'],
                close=data['Close'],
                name=CONFIG['symbol'],
                showlegend=False
            ),
            row=1, col=1
        )
        print("✅ Velas agregadas exitosamente")
    except Exception as e:
        print(f"❌ Error agregando velas: {e}")
        return None
    
    # 2. EMAs
    try:
        data['EMA_21'] = data['Close'].ewm(span=21).mean()
        data['EMA_50'] = data['Close'].ewm(span=50).mean()
        
        fig.add_trace(
            go.Scatter(
                x=data['time'],
                y=data['EMA_21'],
                line=dict(color='orange', width=1.5),
                name='EMA 21'
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=data['time'],
                y=data['EMA_50'],
                line=dict(color='purple', width=1.5),
                name='EMA 50'
            ),
            row=1, col=1
        )
        print("✅ EMAs agregadas exitosamente")
    except Exception as e:
        print(f"⚠️ Error agregando EMAs: {e}")
    
    # 3. SEÑALES ICT CON CARACTERÍSTICAS
    try:
        for i, signal in enumerate(signals):
            signal_time = signal['time']
            signal_price = signal['price']
            
            # Encontrar el índice de la señal
            signal_idx = None
            try:
                signal_mask = data['time'] == signal_time
                if signal_mask.any():
                    signal_idx = data[signal_mask].index[0]
                    signal_idx = data.index.get_loc(signal_idx)
            except:
                continue
            
            if signal_idx is not None and signal_idx >= 20:
                # CARACTERÍSTICAS ICT
                prev_data = data.iloc[signal_idx-20:signal_idx]
                
                if signal['direction'] == 'LONG':
                    # Sweep de máximos
                    prev_high = prev_data['High'].max()
                    
                    # Línea de resistencia barrida
                    fig.add_hline(
                        y=prev_high,
                        line=dict(color='red', width=2, dash='dash'),
                        row=1, col=1
                    )
                    
                    # Zona de sweep
                    fig.add_shape(
                        type="rect",
                        x0=signal_time,
                        x1=signal_time,
                        y0=prev_high,
                        y1=signal_price,
                        fillcolor="red",
                        opacity=0.2,
                        layer="below",
                        line_width=0,
                        row=1, col=1
                    )
                
                else:  # SHORT
                    # Sweep de mínimos
                    prev_low = prev_data['Low'].min()
                    
                    # Línea de soporte barrida
                    fig.add_hline(
                        y=prev_low,
                        line=dict(color='green', width=2, dash='dash'),
                        row=1, col=1
                    )
                    
                    # Zona de sweep
                    fig.add_shape(
                        type="rect",
                        x0=signal_time,
                        x1=signal_time,
                        y0=prev_low,
                        y1=signal_price,
                        fillcolor="green",
                        opacity=0.2,
                        layer="below",
                        line_width=0,
                        row=1, col=1
                    )
                
                # Fakeout (siguiente 5 velas)
                if signal_idx + 5 < len(data):
                    next_candles = data.iloc[signal_idx+1:signal_idx+6]
                    avg_close = next_candles['Close'].mean()
                    
                    # Línea de retorno del fakeout
                    fig.add_scatter(
                        x=[signal_time, next_candles['time'].iloc[-1]],
                        y=[signal_price, avg_close],
                        mode='lines',
                        line=dict(color='orange', width=3, dash='dot'),
                        name=f"Fakeout {signal['direction']}",
                        showlegend=True
                    )
                
                # Volumen anómalo
                current_candle = data.iloc[signal_idx]
                avg_volume = prev_data['tick_volume'].mean()
                volume_ratio = current_candle['tick_volume'] / avg_volume
                
                if volume_ratio > 1.5:
                    fig.add_scatter(
                        x=[signal_time],
                        y=[signal_price],
                        mode='markers',
                        marker=dict(
                            symbol='diamond',
                            size=15,
                            color='purple',
                            line=dict(color='white', width=2)
                        ),
                        name=f"Vol {volume_ratio:.1f}x",
                        showlegend=True
                    )
            
            # SEÑAL PRINCIPAL
            color = '#00FF00' if signal['direction'] == 'LONG' else '#FF0000'
            symbol_marker = 'triangle-up' if signal['direction'] == 'LONG' else 'triangle-down'
            
            fig.add_trace(
                go.Scatter(
                    x=[signal['time']],
                    y=[signal['price']],
                    mode='markers+text',
                    marker=dict(
                        symbol=symbol_marker,
                        size=25,
                        color=color,
                        line=dict(color='white', width=3)
                    ),
                    text=[f"{signal['quality_icon']}"],
                    textposition='middle center',
                    textfont=dict(size=16, color='white'),
                    name=f"{signal['direction']} {signal['quality_tier']} {signal['strength']}%",
                    showlegend=True
                ),
                row=1, col=1
            )
        
        print(f"✅ {len(signals)} señales agregadas exitosamente")
    except Exception as e:
        print(f"⚠️ Error agregando señales: {e}")
    
    # 4. AGREGAR FVG (FAIR VALUE GAPS)
    try:
        if 'fvgs' in SIMPLE_RESULTS and SIMPLE_RESULTS['fvgs']:
            fvgs = SIMPLE_RESULTS['fvgs']
            for fvg in fvgs:
                color = 'rgba(0,255,0,0.3)' if fvg['type'] == 'BULLISH' else 'rgba(255,0,0,0.3)'
                
                # Rectángulo para el FVG
                fig.add_shape(
                    type="rect",
                    x0=fvg['time'],
                    x1=data['time'].iloc[-1],  # Extender hasta el final
                    y0=fvg['low'],
                    y1=fvg['high'],
                    fillcolor=color,
                    opacity=0.4,
                    layer="below",
                    line_width=1,
                    line_color=color.replace('0.3', '0.8'),
                    row=1, col=1
                )
                
                # Etiqueta del FVG
                fig.add_annotation(
                    x=fvg['time'],
                    y=(fvg['high'] + fvg['low']) / 2,
                    text=f"FVG {fvg['type'][:1]}",
                    showarrow=True,
                    arrowhead=2,
                    arrowsize=1,
                    arrowwidth=2,
                    arrowcolor=color.replace('0.3', '1'),
                    ax=20,
                    ay=0,
                    bgcolor="white",
                    bordercolor=color.replace('0.3', '1'),
                    font=dict(size=8)
                )
            print(f"✅ {len(fvgs)} FVGs agregados exitosamente")
    except Exception as e:
        print(f"⚠️ Error agregando FVGs: {e}")
    
    # 5. BPR ELIMINADOS - Solo mantenemos FVGs como solicitado
    
    # 6. AGREGAR MSS (MARKET STRUCTURE SHIFTS) 
    try:
        if 'mss' in SIMPLE_RESULTS and SIMPLE_RESULTS['mss']:
            mss_events = SIMPLE_RESULTS['mss']
            for mss in mss_events:
                color = 'darkgreen' if mss['type'] == 'BULLISH' else 'darkred'
                symbol = 'arrow-up' if mss['type'] == 'BULLISH' else 'arrow-down'
                
                # Marcador para MSS
                fig.add_scatter(
                    x=[mss['time']],
                    y=[data[data['time'] == mss['time']]['High'].iloc[0] if mss['type'] == 'BULLISH' 
                       else data[data['time'] == mss['time']]['Low'].iloc[0]],
                    mode='markers+text',
                    marker=dict(
                        symbol=symbol,
                        size=20,
                        color=color,
                        line=dict(color='white', width=2)
                    ),
                    text=['MSS'],
                    textposition='middle center',
                    textfont=dict(size=10, color='white'),
                    name=f"MSS {mss['type']}",
                    showlegend=True,
                    hovertemplate=f"<b>Market Structure Shift</b><br>" +
                                f"Tipo: {mss['type']}<br>" +
                                f"Descripción: {mss['description']}<br>" +
                                f"Tiempo: %{{x}}<extra></extra>"
                )
            print(f"✅ {len(mss_events)} MSS agregados exitosamente")
    except Exception as e:
        print(f"⚠️ Error agregando MSS: {e}")
    
    # 7. KILLZONES
    try:
        # London Session (8:00-11:00 EST)
        london_times = data[(data['time'].dt.hour >= 8) & (data['time'].dt.hour <= 11)]
        if len(london_times) > 0:
            fig.add_vrect(
                x0=london_times['time'].iloc[0],
                x1=london_times['time'].iloc[-1],
                fillcolor="blue",
                opacity=0.1,
                layer="below",
                line_width=0,
                row=1, col=1
            )
        
        # New York Session (13:30-16:30 EST)
        ny_times = data[(data['time'].dt.hour >= 13) & (data['time'].dt.hour <= 16)]
        if len(ny_times) > 0:
            fig.add_vrect(
                x0=ny_times['time'].iloc[0],
                x1=ny_times['time'].iloc[-1],
                fillcolor="red",
                opacity=0.1,
                layer="below",
                line_width=0,
                row=1, col=1
            )
        print("✅ Killzones agregadas exitosamente")
    except Exception as e:
        print(f"⚠️ Error agregando killzones: {e}")
    
    # 5. VOLUMEN
    try:
        colors = ['green' if close >= open else 'red' 
                  for close, open in zip(data['Close'], data['Open'])]
        
        fig.add_trace(
            go.Bar(
                x=data['time'],
                y=data['tick_volume'],
                name='Volumen',
                marker_color=colors,
                opacity=0.7,
                showlegend=False
            ),
            row=2, col=1
        )
        print("✅ Volumen agregado exitosamente")
    except Exception as e:
        print(f"⚠️ Error agregando volumen: {e}")
    
    # Layout
    fig.update_layout(
        title={
            'text': f"📊 {CONFIG['symbol']} - {CONFIG['formatted_date']} ({CONFIG['day_name']}) - {SIMPLE_RESULTS['classification']}<br>" +
                   f"<sub>🔍 Análisis ICT con Características Técnicas Visualizadas</sub>",
            'x': 0.5,
            'font': {'size': 18, 'color': '#2E86AB'}
        },
        xaxis_rangeslider_visible=False,
        height=900,
        template='plotly_white',
        hovermode='x unified',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="gray",
            borderwidth=1
        )
    )
    
    fig.update_xaxes(title_text="Tiempo (EST)", row=2, col=1)
    fig.update_yaxes(title_text="Precio", row=1, col=1)
    fig.update_yaxes(title_text="Volumen", row=2, col=1)
    
    return fig

# Crear y mostrar gráfico
if SIMPLE_RESULTS:
    fig_simple = create_simple_chart()
    if fig_simple:
        fig_simple.show()
        
        print(f"\n🎯 RESUMEN EJECUTIVO - {CONFIG['formatted_date']}:")
        print("=" * 50)
        print(f"📅 Fecha analizada: {CONFIG['formatted_date']} ({CONFIG['day_name']})")
        print(f"🏆 Clasificación del día: {SIMPLE_RESULTS['classification']}")
        print(f"🎯 Señales encontradas: {len(SIMPLE_RESULTS['signals'])}")
        
        if SIMPLE_RESULTS['signals']:
            avg_strength = sum(s['strength'] for s in SIMPLE_RESULTS['signals']) / len(SIMPLE_RESULTS['signals'])
            print(f"💪 Fuerza promedio: {avg_strength:.1f}%")
            
            stats = SIMPLE_RESULTS['stats']
            print(f"📊 Distribución de calidad:")
            print(f"   💎 Premium: {stats['premium']}")
            print(f"   🔥 High: {stats['high']}")  
            print(f"   ⚡ Medium: {stats['medium']}")
            print(f"   ⚠️ Low: {stats['low']}")
            
            print(f"\n🔍 ELEMENTOS ICT VISUALIZADOS:")
            print("   🎯 Líneas punteadas: Niveles barridos (Sweeps)")
            print("   🎭 Zonas coloreadas: Sweeps detectados") 
            print("   📊 Líneas naranjas: Retorno del fakeout")
            print("   📈 Diamantes morados: Volumen anómalo")
            print("   ⚡ Triángulos grandes: Señales ICT validadas")
            print("   🔶 Rectángulos transparentes: Fair Value Gaps (FVG)")
            print("   📈 Líneas punteadas gruesas: Break Previous Range (BPR)")
            print("   🔄 Flechas MSS: Market Structure Shifts")
            
            # Mostrar estadísticas de elementos ICT
            if 'fvgs' in SIMPLE_RESULTS:
                fvgs = SIMPLE_RESULTS['fvgs']
                print(f"\n📊 ESTADÍSTICAS ICT:")
                print(f"   🔶 FVGs: {len(fvgs)} ({'B:' + str(len([f for f in fvgs if f['type'] == 'BULLISH'])) + ', B:' + str(len([f for f in fvgs if f['type'] == 'BEARISH']))})")
            
            if 'bprs' in SIMPLE_RESULTS:
                bprs = SIMPLE_RESULTS['bprs']
                print(f"   📈 BPRs: {len(bprs)} ({'B:' + str(len([b for b in bprs if b['type'] == 'BULLISH'])) + ', B:' + str(len([b for b in bprs if b['type'] == 'BEARISH']))})")
                
            if 'mss' in SIMPLE_RESULTS:
                mss_events = SIMPLE_RESULTS['mss']
                print(f"   🔄 MSS: {len(mss_events)} ({'B:' + str(len([m for m in mss_events if m['type'] == 'BULLISH'])) + ', B:' + str(len([m for m in mss_events if m['type'] == 'BEARISH']))})")
            
            for i, signal in enumerate(SIMPLE_RESULTS['signals'], 1):
                print(f"\n   📍 Señal {i} - {signal['direction']} {signal['quality_icon']}:")
                print(f"      🕐 {signal['time'].strftime('%H:%M')} | 💰 {signal['price']:,.2f}")
                print(f"      🎯 {signal['type']} | 💪 {signal['strength']}%")
        
        print(f"\n🎨 Gráfico ICT con todas las características técnicas visualizadas")
        
else:
    print("❌ No se pudo generar el gráfico")

🎨 Creando gráfico para 29/09/2025...
📊 Datos disponibles: 35 velas
🎯 Señales disponibles: 0
✅ Velas agregadas exitosamente
✅ EMAs agregadas exitosamente
✅ 0 señales agregadas exitosamente
✅ Killzones agregadas exitosamente
✅ Volumen agregado exitosamente



🎯 RESUMEN EJECUTIVO - 29/09/2025:
📅 Fecha analizada: 29/09/2025 (Monday)
🏆 Clasificación del día: ❌ MALO
🎯 Señales encontradas: 0

🎨 Gráfico ICT con todas las características técnicas visualizadas
